In [1]:
import requests, os
from astropy.time import Time

In [2]:
# This is the archive token which can be obtained from https://ampel.zeuthen.desy.de/live/dashboard/tokens
# In order to retrieve ZTF partnership alerts your token needs to have the appropriate access
# token = os.environ["ARCHIVE_TOKEN"]   # I have mine stored
token = "0bdfc911-21a7-42dd-ae90-65bfcf1f1b99"
header = {"Authorization": "bearer "+token}

In [3]:
endpoint = 'https://ampel.zeuthen.desy.de/api/ztf/archive/v3/streams/from_query?programid=1'

In [4]:
date_str = '2026-03-01' #start of the search
start_jd = Time(date_str,
                format='iso',
                scale="utc",
                ).jd
end_jd = Time.now().jd #until now
print(end_jd)

2461123.9388420093


In [17]:
query = {
"cone": {
    "ra": 149.9287,
    "dec": 0.4177,
    "radius": 10. /3600 #10 arcsec
  },
"candidate": {
    "drb": {
      "$gt": 0.995
    },
    "ndethist": {
      "$gt": 1,
      "$lte": 10000
    },
    "isdiffpos": {"$in": ["t", "1"]},
  },
"jd": {"$gt": start_jd, "$lt": end_jd}
}

In [12]:
response = requests.post(endpoint, headers=header, json=query )
if not response.ok:
    print( 'Query creation failed.')

In [13]:
response.json()

{'resume_token': 'i90mxZ-hF3g9-X_7I7qU6Fv81KgH0UIFWKehXNw8vsY',
 'chunk_size': 100}

In [14]:
resume_token = response.json()['resume_token']

In [15]:
from ampel.ztf.t0.load.ZTFArchiveAlertLoader import ZTFArchiveAlertLoader
# The loader config contains the resume_token as stream identifier
config = {'archive':"https://ampel.zeuthen.desy.de/api/ztf/archive/v3", 
          "stream":resume_token}

In [16]:
try:
    alertlist = []
    alertloader = ZTFArchiveAlertLoader(**config)
    alerts = alertloader.get_alerts()
    for alert in alerts:
        alertlist.append(alert)
        print(alert['objectId'], alert['candidate']['jd'])
except requests.exceptions.HTTPError as e:
    status_code = e.response.status_code
    if status_code==423:
        print('HTTP error {}: likely caused by server staging process. Wait and try again.'.format(status_code) )
    else:
        raise e
      

2026-03-24 11:32:29 ZTFArchiveAlertLoader:110 INFO

Remaining chunks: 0